# Walk through the dashboard

`dashboard.py` only draws the page. The numbers come from `analysis/`.

1. `analysis/prepare.py` sets text, city, and channel.
2. `analysis/stance.py` sets text stance, vendor tone, and the three circles.
3. `analysis/terms.py` supplies the word-cloud terms.
4. `analysis/summary.py` filters, counts the timeline, computes the trend, and reads peak markers.

City, channel, and the circle source live in `analysis/config.py`. Restart the script after a change.

```bash
python dashboard.py
```

Browser: [http://127.0.0.1:8765](http://127.0.0.1:8765)

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
for candidate in (root, root.parent, root / "Hackathon-KJRJ", root.parent / "Hackathon-KJRJ"):
    if (candidate / "dashboard.py").is_file():
        root = candidate
        break
sys.path.insert(0, str(root))

import dashboard as dash

print("Posts file:", dash.POSTS_FILE)
print("Circle source:", dash.STANCE_SOURCE)
print("City buttons:", [entry["id"] for entry in dash.CITY_BUTTONS])
print("Project -> city:")
for project, city in dash.CITY_BY_PROJECT.items():
    print(f"  {project} -> {city}")
print("Source channel -> button:")
for source, button in dash.CHANNEL_BY_SOURCE.items():
    print(f"  {source} -> {button}")

## Load posts

`load_posts` reads `data/01_filtered/posts.json` and adds the columns.

- `city` comes from `CITY_BY_PROJECT` (IPAI stays in Heilbronn)
- `channel` is News or Social. Other web has no button
- `circle` is Support, Neutral, or Opposition
- `stance` comes from the phrase patterns, `vendor_tone` from the `sentiment` field
- `terms` are the word-cloud tokens, with the Stimme paywall notice removed

In [ ]:
posts = dash.load_posts()
print(f"{len(posts)} posts")
print("From", posts["date"].min().date(), "to", posts["date"].max().date())
print("\nCity:")
print(posts["city"].value_counts().to_string())
print("\nChannel:")
print(posts["channel"].value_counts(dropna=False).to_string())
print("\nCircle:")
print(posts["circle"].value_counts(dropna=False).to_string())
dash.overview()

## The same selection as the buttons

`All` drops the city filter. `date_from` and `date_to` are inclusive.
Leave a date as `None` and the dashboard uses the full span of the current city and channel.

In [ ]:
CITY = "Heilbronn"    # All, Heilbronn, Bonn, Paris
CHANNEL = "News"      # News, Social
DATE_FROM = "2025-06-01"
DATE_TO = "2025-12-31"

view = dash.summarize(city=CITY, channel=CHANNEL, date_from=DATE_FROM, date_to=DATE_TO)

print(view["city"], "|", view["channel"])
print("Window:", view["date_from"], "to", view["date_to"])
print("Posts:", view["post_count"])
print("\nCircles:")
for name in dash.CIRCLES:
    count = view["circles"]["counts"][name]
    share = view["circles"]["shares"][name]
    print(f"  {name:<12} {count:4d}   {share:.0%}")
print("Unclassified:", view["circles"]["unclassified"])
print("\nTrend:", view["trend"]["direction"])
print(view["trend"]["sentence"])
print("\nTen terms:")
for entry in view["terms"]:
    print(f"  {entry['count']:4d}  {entry['term']}")
print("\nPeak markers:", len(view["peaks"]))

## The posts behind the number

A click on the post count in the page shows the same rows: date, project, vendor tone, text stance, title.

In [ ]:
window = dash.select_posts(posts, city=CITY, channel=CHANNEL, date_from=DATE_FROM, date_to=DATE_TO)
columns = ["date", "project_label", "channel", "circle", "vendor_tone", "stance", "title", "source_name"]
window.sort_values("date", ascending=False)[columns].head(12)

## What the page calls

| Element | Function |
| --- | --- |
| City and channel buttons | `select_posts` |
| Timeline | `daily_counts` on the selection without the date window |
| Three circles | `count_circles` |
| Arrow | `trend`, compared with the previous window of the same length |
| Count on the right | length of the window; a click loads `list_posts` |
| Word cloud | `top_terms` |
| Rust peak lines | `load_peak_markers`, from `data/02_peaks/peaks.json` |

`summarize` is the packet the browser gets from `/api/summary`.